# Notebook 20: Capstone — ML Security Audit Pipeline

## Why This Matters

Individual security techniques are tools. An audit pipeline is a workflow. The difference matters: a practitioner with fifteen separate notebooks they must run in the right order, on compatible data formats, with manual interpretation of outputs, is not safer than a practitioner with none — they're just busier.

This capstone assembles all nineteen preceding notebooks into a single, callable ML security audit pipeline. Given a model and a dataset, it runs the full gauntlet:

1. **Supply chain check**: Is the model file safe to load? (pickle opcodes, safetensors, ONNX)
2. **Data audit** (`poison-control`): Does the training data contain poisoned samples?
3. **Model behavioral audit** (`model-scan`): Does the model exhibit backdoor behavior?
4. **Privacy audit**: What is the membership inference attack AUC? Would DP training be appropriate?
5. **Threat report**: Consolidated risk assessment with mitigations.

This is the kind of pipeline a security team would run before deploying an ML system in a regulated environment — healthcare, finance, defense — or when onboarding a model from a third-party vendor.

## Background

### The ML Security Audit Framework

The NIST AI Risk Management Framework (AI RMF) and the EU AI Act both call for systematic risk assessments of ML systems. Neither provides specific technical guidance on *how* to perform such assessments. This pipeline is an opinionated implementation of what a technical ML security audit looks like.

The audit follows the attack taxonomy from notebook 1 (STRIDE applied to ML):
- **Training-time attacks**: data poisoning (checked by poison-control)
- **Supply chain attacks**: model file integrity (supply chain scanner)
- **Inference-time attacks**: backdoor behavior (model-scan)
- **Privacy attacks**: membership inference (MI audit)

### Risk Scoring

Each component produces a signal. The pipeline aggregates them into:
- **Per-component risk level**: LOW / MEDIUM / HIGH
- **Overall risk level**: the maximum across components (conservative)
- **Actionable recommendations**: specific mitigations tied to each finding

### What This Pipeline Cannot Do

No static scanner can provide formal guarantees. A sophisticated adversary who knows the scanner's techniques can craft attacks that evade detection. The pipeline provides:
- Detection of naive/opportunistic attacks (standard backdoors, obvious poisoning)
- Baseline measurements (MI AUC, trigger norms) for comparison
- A documented audit trail for compliance purposes

Against adaptive adversaries or zero-day attack techniques, additional defenses (DP training, robust aggregation, formal verification) are required.

## Structure of This Notebook

1. Scenario setup: three models under audit (clean, backdoored, poisoned)
2. Component 1: Supply chain scanner
3. Component 2: Data audit (poison-control lite)
4. Component 3: Behavioral audit (model-scan lite)
5. Component 4: Privacy audit (membership inference)
6. Audit report generation
7. Applying the pipeline to all three scenarios
8. Series retrospective: what we covered across 20 notebooks

## What You'll Learn

- How to compose multiple security tools into a coherent audit pipeline
- How to translate raw scanner outputs into risk levels and mitigations
- The structure of a formal ML security audit report
- Which techniques from the series apply to which threat categories

In [ ]:
%pip install torch torchvision numpy matplotlib scikit-learn --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Subset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, roc_auc_score
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import NearestNeighbors
import pickle
import io
import copy
import time
from datetime import datetime

device = (
    torch.device('mps') if torch.backends.mps.is_available()
    else torch.device('cuda') if torch.cuda.is_available()
    else torch.device('cpu')
)
print(f'Device: {device}')
torch.manual_seed(42)
np.random.seed(42)

## 1. Setup — Three Audit Targets

We create three models representing realistic scenarios:
- **Clean model**: No issues (expected: all LOW risk)
- **Backdoored model**: BadNets patch trigger (expected: HIGH behavioral risk)
- **Memorizing model**: Trained to overfit training data (expected: HIGH privacy risk)

In [ ]:
class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.drop = nn.Dropout(0.25)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        return self.fc2(x)

    def get_embeddings(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        return F.relu(self.fc1(x))


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_dataset = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

BACKDOOR_TARGET = 0
PATCH_SIZE = 4

def apply_trigger(x):
    x = x.clone()
    x[..., -PATCH_SIZE:, -PATCH_SIZE:] = 2.0
    return x


def quick_train(model, loader, n_epochs=5, lr=1e-3, verbose=False):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(n_epochs):
        model.train()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            F.cross_entropy(model(x), y).backward()
            opt.step()
        if verbose:
            model.eval()
            with torch.no_grad():
                acc = sum(
                    (model(x.to(device)).argmax(1) == y.to(device)).sum().item()
                    for x, y in test_loader
                ) / len(test_dataset)
            print(f'  Epoch {epoch+1}: test_acc={acc:.4f}')


def get_test_accuracy(model):
    model.eval()
    correct = sum(
        (model(x.to(device)).argmax(1) == y.to(device)).sum().item()
        for x, y in test_loader
    )
    return correct / len(test_dataset)


# Model 1: Clean
print('Training Model 1: Clean...')
model_clean = MnistCNN().to(device)
quick_train(model_clean, train_loader, n_epochs=5, verbose=True)
acc_clean = get_test_accuracy(model_clean)
print(f'Clean model accuracy: {acc_clean:.4f}')

# Model 2: Backdoored
print('\nTraining Model 2: Backdoored (10% poison)...')
all_imgs = torch.stack([train_dataset[i][0] for i in range(len(train_dataset))])
all_labels = torch.tensor([train_dataset[i][1] for i in range(len(train_dataset))])
n_poison = int(len(all_imgs) * 0.1)
p_idx = np.random.choice(len(all_imgs), n_poison, replace=False)
all_imgs_bd = all_imgs.clone()
all_imgs_bd[p_idx] = apply_trigger(all_imgs_bd[p_idx])
all_labels_bd = all_labels.clone()
all_labels_bd[p_idx] = BACKDOOR_TARGET

bd_loader = DataLoader(TensorDataset(all_imgs_bd, all_labels_bd), batch_size=128, shuffle=True)
model_backdoored = MnistCNN().to(device)
quick_train(model_backdoored, bd_loader, n_epochs=5)
acc_backdoored = get_test_accuracy(model_backdoored)
print(f'Backdoored model accuracy: {acc_backdoored:.4f}')

# Model 3: Memorizing (overfit on small dataset)
print('\nTraining Model 3: Memorizing (small dataset, no regularization)...')
small_subset = Subset(train_dataset, range(2000))
small_loader = DataLoader(small_subset, batch_size=32, shuffle=True)
model_memorizing = MnistCNN().to(device)
# Train for many epochs to force memorization
quick_train(model_memorizing, small_loader, n_epochs=20)
acc_memorizing = get_test_accuracy(model_memorizing)
print(f'Memorizing model accuracy: {acc_memorizing:.4f}')

## 2. Component 1: Supply Chain Scanner

In [ ]:
import pickletools
import hashlib

DANGEROUS_GLOBALS = {
    'os.system', 'os.popen', 'subprocess.Popen', 'subprocess.call',
    'eval', 'exec', 'compile', '__import__',
    'builtins.eval', 'builtins.exec',
}
ALLOWED_PREFIXES = ('torch', 'collections', '_codecs', 'numpy', '_numpy')


def scan_model_bytes(model_bytes):
    """
    Scan serialized model bytes for dangerous pickle opcodes.
    Returns: (is_safe, findings)
    """
    findings = []
    try:
        ops = list(pickletools.genops(io.BytesIO(model_bytes)))
    except Exception as e:
        return False, [f'Failed to parse pickle: {e}']

    for opcode, arg, pos in ops:
        if opcode.name == 'GLOBAL' and arg:
            module_name = str(arg)
            if any(d in module_name for d in ['os.', 'subprocess.', 'eval', 'exec']):
                findings.append(f'DANGEROUS GLOBAL at pos {pos}: {module_name}')
            elif not any(module_name.startswith(p) for p in ALLOWED_PREFIXES):
                findings.append(f'UNUSUAL GLOBAL at pos {pos}: {module_name}')

    is_safe = not any('DANGEROUS' in f for f in findings)
    return is_safe, findings


def compute_sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def supply_chain_scan(model):
    """
    Serialize model to bytes and scan for supply chain risks.
    Returns supply chain risk report.
    """
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    model_bytes = buf.getvalue()

    is_safe, findings = scan_model_bytes(model_bytes)
    sha256 = compute_sha256_bytes(model_bytes)
    size_kb = len(model_bytes) / 1024

    dangerous_count = sum(1 for f in findings if 'DANGEROUS' in f)
    unusual_count = sum(1 for f in findings if 'UNUSUAL' in f)

    risk = 'HIGH' if dangerous_count > 0 else 'MEDIUM' if unusual_count > 2 else 'LOW'

    return {
        'component': 'supply_chain',
        'risk_level': risk,
        'is_safe': is_safe,
        'sha256': sha256[:16] + '...',
        'size_kb': size_kb,
        'dangerous_globals': dangerous_count,
        'unusual_globals': unusual_count,
        'findings': findings[:5] if findings else ['No dangerous opcodes found'],
    }


print('Supply chain scanner initialized.')
test_result = supply_chain_scan(model_clean)
print(f'Test scan (clean model): risk={test_result["risk_level"]}, '
      f'dangerous={test_result["dangerous_globals"]}, '
      f'sha256={test_result["sha256"]}')

## 3. Component 2: Data Audit (poison-control lite)

In [ ]:
def data_audit(model, train_imgs, train_labels, n_samples=2000, contamination=0.1):
    """
    Lightweight poison-control scan:
    - Isolation Forest on embeddings
    - Label-prediction disagreement
    - k-NN label consistency
    Returns: risk level and top suspicious indices.
    """
    model.eval()

    # Sample for speed
    idx = np.random.choice(len(train_imgs), min(n_samples, len(train_imgs)), replace=False)
    imgs_s = train_imgs[idx]
    labels_s = train_labels[idx]

    # Extract embeddings
    with torch.no_grad():
        embeds = []
        for i in range(0, len(imgs_s), 256):
            batch = imgs_s[i:i+256].to(device)
            embeds.append(model.get_embeddings(batch).cpu().numpy())
        embeds = np.vstack(embeds)

    # Signal 1: Isolation Forest
    iso = IsolationForest(contamination=contamination, random_state=42)
    iso_scores = -iso.fit(embeds).decision_function(embeds)

    # Signal 2: Training loss
    losses = []
    with torch.no_grad():
        for i in range(0, len(imgs_s), 256):
            x = imgs_s[i:i+256].to(device)
            y = labels_s[i:i+256].to(device)
            loss = F.cross_entropy(model(x), y, reduction='none')
            losses.extend(loss.cpu().numpy())
    loss_scores = np.array(losses)

    # Signal 3: k-NN label inconsistency
    nbrs = NearestNeighbors(n_neighbors=6).fit(embeds)
    _, indices = nbrs.kneighbors(embeds)
    labels_np = labels_s.numpy()
    knn_inc = np.array([
        (labels_np[indices[i, 1:]] != labels_np[i]).mean()
        for i in range(len(labels_np))
    ])

    # Composite score
    def norm01(x):
        return (x - x.min()) / (x.max() - x.min() + 1e-8)

    composite = 0.35 * norm01(iso_scores) + 0.35 * norm01(loss_scores) + 0.3 * norm01(knn_inc)

    # Risk level: what fraction of top suspects are statistically anomalous?
    threshold = np.percentile(composite, 90)
    n_flagged = (composite >= threshold).sum()
    flagged_frac = n_flagged / len(composite)

    risk = 'HIGH' if flagged_frac > 0.12 else 'MEDIUM' if flagged_frac > 0.08 else 'LOW'

    top_suspects = idx[np.argsort(-composite)[:10]].tolist()

    return {
        'component': 'data_audit',
        'risk_level': risk,
        'n_scanned': n_samples,
        'n_flagged': int(n_flagged),
        'flagged_fraction': float(flagged_frac),
        'top_suspects': top_suspects,
        'max_composite_score': float(composite.max()),
        'composite_scores': composite,
        'sampled_idx': idx,
    }


print('Data audit initialized.')

## 4. Component 3: Behavioral Audit (model-scan lite)

In [ ]:
def nc_scan_fast(model, probe_loader, n_classes=10, n_steps=80):
    """Fast Neural Cleanse variant (fewer steps for pipeline use)."""
    model.eval()
    norms = []
    for target in range(n_classes):
        mask_raw = torch.zeros(1, 1, 28, 28, device=device, requires_grad=True)
        pattern = torch.zeros(1, 1, 28, 28, device=device, requires_grad=True)
        opt = torch.optim.Adam([mask_raw, pattern], lr=0.1)
        data_iter = iter(probe_loader)
        for step in range(n_steps):
            try:
                x_batch, _ = next(data_iter)
            except StopIteration:
                data_iter = iter(probe_loader)
                x_batch, _ = next(data_iter)
            x_batch = x_batch[:16].to(device)
            mask = torch.sigmoid(mask_raw)
            x_trig = x_batch * (1 - mask) + torch.tanh(pattern) * mask
            opt.zero_grad()
            y_t = torch.full((len(x_batch),), target, dtype=torch.long, device=device)
            loss = F.cross_entropy(model(x_trig), y_t) + 0.01 * mask.sum()
            loss.backward()
            opt.step()
        with torch.no_grad():
            norms.append(torch.sigmoid(mask_raw).sum().item())
    norms = np.array(norms)
    med = np.median(norms)
    mad = np.median(np.abs(norms - med))
    anomaly = np.abs(norms - med) / (mad + 1e-8)
    flagged = [c for c in range(n_classes) if anomaly[c] > 2.0]
    return norms, anomaly, flagged


def strip_scan(model, test_dataset, n_samples=80, n_perturb=10):
    """Quick STRIP entropy baseline."""
    model.eval()
    ref_imgs = torch.stack([test_dataset[i][0] for i in range(100)])
    entropies = []
    with torch.no_grad():
        for i in range(n_samples):
            x, _ = test_dataset[i]
            x = x.unsqueeze(0).to(device)
            hs = []
            for _ in range(n_perturb):
                ref = ref_imgs[np.random.randint(len(ref_imgs))].to(device)
                blended = 0.5 * x + 0.5 * ref.unsqueeze(0)
                probs = F.softmax(model(blended), dim=-1)
                h = -(probs * (probs + 1e-10).log()).sum().item()
                hs.append(h)
            entropies.append(np.mean(hs))
    return float(np.mean(entropies)), float(np.std(entropies))


def behavioral_audit(model, probe_loader, test_dataset, n_classes=10):
    """
    Behavioral backdoor scan:
    - Neural Cleanse trigger norm analysis
    - STRIP entropy baseline
    """
    nc_norms, nc_anomaly, nc_flagged = nc_scan_fast(model, probe_loader, n_classes)
    strip_mean, strip_std = strip_scan(model, test_dataset, n_samples=60, n_perturb=10)

    nc_target = int(np.argmin(nc_norms))
    risk = 'HIGH' if len(nc_flagged) > 0 else 'LOW'

    return {
        'component': 'behavioral_audit',
        'risk_level': risk,
        'nc_flagged_classes': nc_flagged,
        'nc_suspected_target': nc_target,
        'nc_min_norm': float(nc_norms.min()),
        'nc_max_anomaly': float(nc_anomaly.max()),
        'strip_entropy_mean': strip_mean,
        'strip_entropy_std': strip_std,
        'trigger_norms': nc_norms.tolist(),
    }


print('Behavioral audit initialized.')

## 5. Component 4: Privacy Audit (Membership Inference)

In [ ]:
def privacy_audit(model, member_loader, nonmember_loader):
    """
    Membership inference audit using loss-based threshold attack.
    Returns MI AUC and risk level.
    """
    model.eval()
    scores = []
    labels = []

    with torch.no_grad():
        for x, y in member_loader:
            x, y = x.to(device), y.to(device)
            loss = F.cross_entropy(model(x), y, reduction='none')
            scores.extend((-loss).cpu().numpy())
            labels.extend([1] * len(x))

        for x, y in nonmember_loader:
            x, y = x.to(device), y.to(device)
            loss = F.cross_entropy(model(x), y, reduction='none')
            scores.extend((-loss).cpu().numpy())
            labels.extend([0] * len(x))

    mi_auc = roc_auc_score(labels, scores)

    # Risk level based on MI AUC
    # 0.5 = random (ideal), 1.0 = perfect attack (worst)
    if mi_auc >= 0.75:
        risk = 'HIGH'
        advice = 'Consider DP-SGD training (notebook 14) or reducing model capacity'
    elif mi_auc >= 0.60:
        risk = 'MEDIUM'
        advice = 'Model shows moderate memorization; add regularization or early stopping'
    else:
        risk = 'LOW'
        advice = 'Low memorization risk; MI attack not significantly above random'

    return {
        'component': 'privacy_audit',
        'risk_level': risk,
        'mi_auc': float(mi_auc),
        'advice': advice,
        'n_members': sum(1 for l in labels if l == 1),
        'n_nonmembers': sum(1 for l in labels if l == 0),
    }


print('Privacy audit initialized.')

## 6. Full Audit Pipeline

In [ ]:
def ml_security_audit(
    model,
    model_name: str,
    train_imgs: torch.Tensor,
    train_labels: torch.Tensor,
    test_dataset,
    n_data_samples: int = 1500,
    run_behavioral: bool = True,
):
    """
    Full ML security audit pipeline.

    Runs:
    1. Supply chain scan
    2. Data audit (poison-control)
    3. Behavioral audit (model-scan) [optional]
    4. Privacy audit (membership inference)

    Returns comprehensive audit report.
    """
    start_time = time.time()
    audit_date = datetime.now().strftime('%Y-%m-%d %H:%M')

    print(f'[AUDIT] {model_name} — starting audit...')

    # Prepare loaders
    probe_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)
    member_loader = DataLoader(
        TensorDataset(train_imgs[:1000], train_labels[:1000]),
        batch_size=256, shuffle=False
    )
    nonmember_loader = DataLoader(
        Subset(test_dataset, range(1000)), batch_size=256, shuffle=False
    )

    results = {'model_name': model_name, 'audit_date': audit_date}

    # Component 1: Supply chain
    print(f'  [1/4] Supply chain scan...')
    results['supply_chain'] = supply_chain_scan(model)

    # Component 2: Data audit
    print(f'  [2/4] Data audit (poison-control)...')
    results['data_audit'] = data_audit(
        model, train_imgs, train_labels, n_samples=n_data_samples
    )

    # Component 3: Behavioral audit
    if run_behavioral:
        print(f'  [3/4] Behavioral audit (model-scan)...')
        results['behavioral'] = behavioral_audit(model, probe_loader, test_dataset)
    else:
        results['behavioral'] = {'component': 'behavioral_audit', 'risk_level': 'SKIPPED'}

    # Component 4: Privacy audit
    print(f'  [4/4] Privacy audit (membership inference)...')
    results['privacy'] = privacy_audit(model, member_loader, nonmember_loader)

    # Overall risk
    risk_levels = [
        results['supply_chain']['risk_level'],
        results['data_audit']['risk_level'],
        results['privacy']['risk_level'],
    ]
    if run_behavioral:
        risk_levels.append(results['behavioral']['risk_level'])

    priority = {'HIGH': 3, 'MEDIUM': 2, 'LOW': 1, 'SKIPPED': 0}
    overall = max(risk_levels, key=lambda r: priority.get(r, 0))
    results['overall_risk'] = overall
    results['elapsed_seconds'] = round(time.time() - start_time, 1)

    print(f'[AUDIT] Complete. Overall risk: {overall} ({results["elapsed_seconds"]}s)')
    return results


def print_audit_report(results):
    """Print a structured audit report."""
    print('\n' + '='*65)
    print(f'  ML SECURITY AUDIT REPORT')
    print(f'  Model: {results["model_name"]}')
    print(f'  Date: {results["audit_date"]}')
    print(f'  Overall Risk: {results["overall_risk"]}')
    print('='*65)

    sc = results['supply_chain']
    print(f'\n  [Supply Chain]  Risk: {sc["risk_level"]}')
    print(f'    SHA256: {sc["sha256"]}  Size: {sc["size_kb"]:.0f}KB')
    print(f'    Dangerous globals: {sc["dangerous_globals"]}  '
          f'Unusual globals: {sc["unusual_globals"]}')
    if sc['dangerous_globals'] > 0:
        print(f'    *** CRITICAL: {sc["findings"][0]} ***')

    da = results['data_audit']
    print(f'\n  [Data Audit]  Risk: {da["risk_level"]}')
    print(f'    Scanned: {da["n_scanned"]} samples | Flagged: {da["n_flagged"]} '
          f'({da["flagged_fraction"]*100:.1f}%)')
    print(f'    Top suspects: {da["top_suspects"][:5]}')

    beh = results['behavioral']
    if beh['risk_level'] != 'SKIPPED':
        print(f'\n  [Behavioral]  Risk: {beh["risk_level"]}')
        print(f'    NC flagged classes: {beh["nc_flagged_classes"]}')
        print(f'    NC suspected target: class {beh["nc_suspected_target"]} '
              f'(min norm={beh["nc_min_norm"]:.2f})')
        print(f'    STRIP entropy: mean={beh["strip_entropy_mean"]:.4f}')

    priv = results['privacy']
    print(f'\n  [Privacy]  Risk: {priv["risk_level"]}')
    print(f'    MI AUC: {priv["mi_auc"]:.4f} (0.5=safe, 1.0=fully memorized)')
    print(f'    Advice: {priv["advice"]}')

    print('\n  Mitigations:')
    mitigations = set()
    if results['supply_chain']['risk_level'] != 'LOW':
        mitigations.add('  - Use safetensors format for model distribution (notebook 11)')
    if results['data_audit']['risk_level'] != 'LOW':
        mitigations.add('  - Run poison-control full scan; remove flagged samples (notebook 18)')
    if beh.get('risk_level') == 'HIGH':
        mitigations.add('  - Apply Neural Cleanse unlearning to flagged class (notebook 9)')
        mitigations.add('  - Retrain from scratch on verified clean data')
    if results['privacy']['risk_level'] != 'LOW':
        mitigations.add('  - Apply DP-SGD with noise_multiplier=1.0 (notebook 14)')
        mitigations.add('  - Reduce training epochs to limit memorization')

    if mitigations:
        for m in sorted(mitigations):
            print(m)
    else:
        print('  - No critical issues found. Continue monitoring.')

    print('='*65)


print('Audit pipeline assembled. Running audits...')

## 7. Running the Audit on All Three Models

In [ ]:
# Prepare training data tensors
train_imgs_all = torch.stack([train_dataset[i][0] for i in range(5000)])
train_labels_all = torch.tensor([train_dataset[i][1] for i in range(5000)])

# Backdoored training data
train_imgs_bd_5k = all_imgs_bd[:5000]
train_labels_bd_5k = all_labels_bd[:5000]

# Memorizing model training data (small subset)
train_imgs_small = torch.stack([train_dataset[i][0] for i in range(2000)])
train_labels_small = torch.tensor([train_dataset[i][1] for i in range(2000)])

print('\n' + '#'*65)
print('AUDIT 1: Clean Model')
print('#'*65)
report_clean = ml_security_audit(
    model_clean, 'Clean Model',
    train_imgs_all, train_labels_all,
    test_dataset, n_data_samples=1000, run_behavioral=True
)
print_audit_report(report_clean)

In [ ]:
print('\n' + '#'*65)
print('AUDIT 2: Backdoored Model')
print('#'*65)
report_bd = ml_security_audit(
    model_backdoored, 'Backdoored Model (BadNets)',
    train_imgs_bd_5k, train_labels_bd_5k,
    test_dataset, n_data_samples=1000, run_behavioral=True
)
print_audit_report(report_bd)

In [ ]:
print('\n' + '#'*65)
print('AUDIT 3: Memorizing Model')
print('#'*65)
report_mem = ml_security_audit(
    model_memorizing, 'Memorizing Model (overfit)',
    train_imgs_small, train_labels_small,
    test_dataset, n_data_samples=800, run_behavioral=True
)
print_audit_report(report_mem)

## 8. Risk Comparison Dashboard

In [ ]:
def risk_to_num(r):
    return {'HIGH': 3, 'MEDIUM': 2, 'LOW': 1, 'SKIPPED': 0}.get(r, 0)


models = ['Clean', 'Backdoored', 'Memorizing']
reports = [report_clean, report_bd, report_mem]
components = ['supply_chain', 'data_audit', 'behavioral', 'privacy']
component_names = ['Supply Chain', 'Data Audit', 'Behavioral', 'Privacy']

risk_matrix = np.zeros((len(models), len(components)))
for i, report in enumerate(reports):
    for j, comp in enumerate(components):
        rl = report.get(comp, {}).get('risk_level', 'SKIPPED')
        risk_matrix[i, j] = risk_to_num(rl)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
im = axes[0].imshow(risk_matrix, cmap='RdYlGn_r', vmin=0, vmax=3, aspect='auto')
axes[0].set_xticks(range(len(components)))
axes[0].set_xticklabels(component_names)
axes[0].set_yticks(range(len(models)))
axes[0].set_yticklabels(models)
axes[0].set_title('Risk Level Heatmap (3=HIGH, 2=MEDIUM, 1=LOW)')
for i in range(len(models)):
    for j in range(len(components)):
        val = {3: 'HIGH', 2: 'MED', 1: 'LOW', 0: '-'}[int(risk_matrix[i, j])]
        axes[0].text(j, i, val, ha='center', va='center', fontweight='bold', fontsize=10)
plt.colorbar(im, ax=axes[0])

# MI AUC comparison
mi_aucs = [r['privacy']['mi_auc'] for r in reports]
bar_colors = ['green' if a < 0.60 else 'orange' if a < 0.75 else 'red' for a in mi_aucs]
bars = axes[1].bar(models, mi_aucs, color=bar_colors, edgecolor='black', width=0.4)
axes[1].axhline(0.5, color='green', linestyle='--', alpha=0.5, label='Ideal (random)')
axes[1].axhline(0.75, color='red', linestyle='--', alpha=0.5, label='HIGH risk threshold')
for bar, auc in zip(bars, mi_aucs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{auc:.3f}', ha='center', va='bottom', fontweight='bold')
axes[1].set_ylabel('MI Attack AUC')
axes[1].set_title('Membership Inference AUC')
axes[1].legend()
axes[1].set_ylim(0.4, 1.0)
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('ML Security Audit: Risk Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Audit complete.')
print(f'  Clean model:      {report_clean["overall_risk"]} (elapsed: {report_clean["elapsed_seconds"]}s)')
print(f'  Backdoored model: {report_bd["overall_risk"]} (elapsed: {report_bd["elapsed_seconds"]}s)')
print(f'  Memorizing model: {report_mem["overall_risk"]} (elapsed: {report_mem["elapsed_seconds"]}s)')

## 9. Series Retrospective — 20 Notebooks in Context

In [ ]:
series_map = [
    ('01', 'ML Threat Modeling', 'Framework', 'All phases'),
    ('02', 'FGSM', 'Attack', 'Inference'),
    ('03', 'PGD & C&W', 'Attack', 'Inference'),
    ('04', 'Adversarial Defenses', 'Defense', 'Inference'),
    ('05', 'Membership Inference', 'Attack', 'Privacy'),
    ('06', 'Model Inversion', 'Attack', 'Privacy'),
    ('07', 'Attribute Inference', 'Attack', 'Privacy'),
    ('08', 'Backdoor Attacks', 'Attack', 'Training'),
    ('09', 'Backdoor Detection', 'Defense', 'Training'),
    ('10', 'Data Poisoning', 'Attack', 'Training'),
    ('11', 'Supply Chain Attacks', 'Attack', 'Supply Chain'),
    ('12', 'Model Stealing', 'Attack', 'Inference'),
    ('13', 'DP Fundamentals', 'Defense', 'Privacy'),
    ('14', 'DP-SGD', 'Defense', 'Privacy'),
    ('15', 'Federated Learning', 'Defense', 'Privacy'),
    ('16', 'FL Attacks', 'Attack', 'Training'),
    ('17', 'Model Watermarking', 'Defense', 'Supply Chain'),
    ('18', 'poison-control', 'Tool', 'Training'),
    ('19', 'model-scan', 'Tool', 'Training'),
    ('20', 'Capstone Pipeline', 'Tool', 'All phases'),
]

print('='*65)
print('ML SECURITY SERIES — 20 NOTEBOOKS')
print('='*65)
print(f'{"NB":>4} {"Topic":>35} {"Type":>8} {"Phase":>15}')
print('-'*65)

current_phase = None
for nb, topic, typ, phase in series_map:
    type_colors = {'Attack': '[ATK]', 'Defense': '[DEF]', 'Tool': '[TL]', 'Framework': '[FWK]'}
    print(f'  {nb:>2}  {topic:<35} {type_colors[typ]:<8} {phase}')

print()
print('Attack taxonomy coverage:')
phases = {}
for _, _, typ, phase in series_map:
    if phase not in phases:
        phases[phase] = {'Attack': 0, 'Defense': 0, 'Tool': 0}
    if typ in phases[phase]:
        phases[phase][typ] += 1

for phase, counts in sorted(phases.items()):
    total = sum(counts.values())
    print(f'  {phase:<15}: {counts["Attack"]} attacks, '
          f'{counts["Defense"]} defenses, {counts["Tool"]} tools')

## Summary — Series Capstone

### What the Audit Pipeline Does

| Component | What It Checks | Key Signal | Reference |
|-----------|---------------|------------|-----------|
| Supply chain scan | Model file safety | Dangerous pickle opcodes | Notebook 11 |
| Data audit | Training data integrity | Isolation Forest + influence + k-NN | Notebook 18 |
| Behavioral audit | Model backdoor behavior | NC trigger norms + STRIP entropy | Notebooks 9, 19 |
| Privacy audit | Membership inference risk | MI AUC (loss-based threshold) | Notebook 5 |
| Consolidated report | Overall risk + mitigations | Max risk across components | Notebook 1 |

### Series Coverage — 20 Notebooks

**Training-time attacks & defenses** (Notebooks 8–10, 15–16, 18):
- BadNets, blended triggers, clean-label attacks
- Neural Cleanse, STRIP, activation clustering
- Federated learning attacks (Byzantine, scaling, gradient inversion)
- `poison-control` data scanner

**Inference-time attacks & defenses** (Notebooks 2–4, 12):
- FGSM, PGD, C&W, AutoAttack
- Certified defense (randomized smoothing), TRADES, adversarial training
- Gradient masking diagnostics (monotonic steps, black-box vs white-box)
- Model stealing (Knockoff Nets, adaptive extraction, watermarking)

**Privacy attacks & defenses** (Notebooks 5–7, 13–15):
- Membership inference (threshold, shadow model, LiRA)
- Model inversion (gradient-based, GMI, NES black-box)
- Attribute inference (proxy variables, adversarial debiasing)
- Differential privacy (ε-δ DP, Laplace, Gaussian, composition)
- DP-SGD (per-sample gradient clipping + Gaussian noise)
- Federated learning (FedAvg, FedProx, non-IID convergence)

**Supply chain attacks & defenses** (Notebooks 11, 17):
- Pickle exploits, safetensors format, ONNX graph injection
- Backdoor-based watermarking, weights-based watermarking
- Statistical watermark verification (binomial hypothesis test)

**Tools & frameworks** (Notebooks 1, 18–20):
- STRIDE applied to ML, RAG pipeline threat model
- `poison-control`: training data scanner
- `model-scan`: behavioral backdoor scanner
- Capstone: end-to-end audit pipeline

### The Big Picture

ML security is not a solved problem. Every defense in this series has known bypasses. Every detection method can be evaded by an adversary who knows it's deployed. The state of the art is an asymmetric arms race: defenders must protect the entire attack surface, attackers need only find one gap.

What the series provides is a *vocabulary and toolkit*: the ability to identify which attack category applies to a given threat, which defenses are appropriate, how to implement them, and how to audit a system for known vulnerabilities. This is the foundation for building ML systems that are robust, private, and trustworthy — even if not perfectly secure.

**Formal guarantees remain the gold standard:**
- Differential privacy provides a mathematical bound on privacy leakage
- Certified robustness (randomized smoothing) provides a provable adversarial radius
- Byzantine fault tolerance with formal breakdown point analysis

These formal guarantees have utility costs — but for high-stakes ML deployments, they are the only defensible position against sophisticated adversaries.